# Notebook 7 — Synthetic Pathology Validation

---

## The Problem With Unsupervised Validation

Notebooks 1-6 show that k-space fingerprints are sensitive to acquisition differences, gadolinium enhancement, and outlier scans. But none of that is *verified*. Without ground truth labels, we can't know whether the patterns we're detecting correspond to real clinical findings.

The standard solution is clinical labels — a radiologist reads the scan and says "tumor present" or "normal." But that requires expert clinical judgment that most researchers don't have access to.

**Synthetic pathology sidesteps this entirely.**

---

## The Approach

Instead of finding pathology in the data, we *create* it — mathematically, with known ground truth.

```
Step 1: Load a clean brain scan → reconstruct image
Step 2: Add a synthetic lesion at a known location (Gaussian intensity blob)
Step 3: Apply FFT to convert the modified image back to k-space
Step 4: Compute fingerprint metrics on both original and modified k-space
Step 5: Measure how much the fingerprint changed
Step 6: Vary lesion size and intensity → find the detection threshold
```

The ground truth is mathematical. We know exactly where the lesion is, how large it is, and how intense it is — because we put it there. No radiological judgment required.

---

## What This Proves

If the fingerprint metrics change detectably when a synthetic lesion is added:
- The metrics ARE sensitive to local tissue changes
- The sensitivity scales with lesion size and intensity
- There is a minimum detectable lesion size — a threshold below which k-space can't distinguish pathology from noise

This is the same validation framework used in medical imaging AI research before clinical data is available. You prove the method works on controlled synthetic data, then validate on real clinical data (fastMRI+ annotations).

---

## Step 1 — Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, root_sum_of_squares
from kode.fingerprint import radial_profile, energy_ratio, asymmetry_score

---

## Step 2 — Load a Clean Scan and Reconstruct

We load a single slice and reconstruct it to image space. This is our clean baseline — the scan before any pathology is introduced.

In [ ]:
H5_FILE = '../data/multicoil_test/file_brain_AXFLAIR_200_6002527.h5'
SLICE_IDX = 8

kspace_clean = load_kspace(H5_FILE, slice_idx=SLICE_IDX)
image_clean = root_sum_of_squares(kspace_clean)

print(f'K-space shape: {kspace_clean.shape}')
print(f'Image shape:   {image_clean.shape}')
print(f'Image range:   {image_clean.min():.2f} – {image_clean.max():.2f}')

---

## Step 3 — Define the Synthetic Lesion

A lesion is added as a **Gaussian intensity blob** — a smooth circular region of elevated signal, placed at a specific pixel coordinate in the image.

Why Gaussian? Because real lesions don't have sharp edges — they fade into surrounding tissue. A Gaussian approximates that gradual boundary realistically.

Parameters you control:
- `center_y`, `center_x` — where in the image the lesion is placed
- `radius` — how large the lesion is (in pixels)
- `intensity` — how bright the lesion is relative to surrounding tissue (0.0 = invisible, 1.0 = as bright as the brightest point in the image)

In [ ]:
def add_synthetic_lesion(image, center_y, center_x, radius, intensity):
    """
    Add a Gaussian intensity lesion to a 2D image at a known location.
    Returns the modified image — ground truth location is (center_y, center_x).
    """
    H, W = image.shape
    y, x = np.ogrid[:H, :W]
    dist = np.sqrt((y - center_y)**2 + (x - center_x)**2)
    gaussian = np.exp(-dist**2 / (2 * radius**2))
    lesion_signal = gaussian * intensity * image.max()
    return image + lesion_signal


def image_to_kspace(image):
    """
    Convert a 2D image back to k-space via FFT.
    Returns a single-coil k-space array shape (1, H, W).
    """
    kspace_2d = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image)))
    return kspace_2d[np.newaxis, :, :]  # add coil dim


H, W = image_clean.shape
# Place lesion in the right hemisphere — known ground truth location
CENTER_Y = H // 2
CENTER_X = W // 2 + W // 4
RADIUS = 15
INTENSITY = 0.5

image_lesion = add_synthetic_lesion(image_clean, CENTER_Y, CENTER_X, RADIUS, INTENSITY)
kspace_lesion = image_to_kspace(image_lesion)

print(f'Lesion placed at pixel ({CENTER_Y}, {CENTER_X})')
print(f'Lesion radius: {RADIUS}px  |  Intensity: {INTENSITY} × image max')

---

## Step 4 — Visualize Clean vs Lesion

Three panels:
1. Clean image — what the scan looks like without pathology
2. Image with synthetic lesion — the bright spot at the known location
3. Difference map — exactly where the lesion changed the image (should show a clean Gaussian blob at the expected location)

The difference map is the ground truth visualization — it proves the lesion is exactly where we said it would be.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_clean, cmap='gray')
axes[0].set_title('Clean Scan\n(no pathology)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(image_lesion, cmap='gray')
axes[1].scatter([CENTER_X], [CENTER_Y], c='red', s=80, marker='x', linewidths=2, label='Lesion center')
axes[1].set_title(f'Synthetic Lesion Added\n(radius={RADIUS}px, intensity={INTENSITY})', fontsize=11)
axes[1].legend(loc='lower right')
axes[1].axis('off')

diff = image_lesion - image_clean
axes[2].imshow(diff, cmap='hot')
axes[2].scatter([CENTER_X], [CENTER_Y], c='cyan', s=80, marker='x', linewidths=2, label='Ground truth location')
axes[2].set_title('Difference Map\n(lesion signal only)', fontsize=11)
axes[2].legend(loc='lower right')
axes[2].axis('off')

plt.suptitle('Synthetic Pathology — Ground Truth is Mathematical', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../results/synthetic_lesion_visual.png', dpi=150)
plt.show()

**What this demonstrates:**

The difference map confirms the lesion is exactly where we placed it — a clean Gaussian blob at the known pixel coordinate. This is the ground truth. No radiologist needed because we created it mathematically. The next step is to ask: can k-space fingerprint metrics detect this change?

---

## Step 5 — Compare K-Space Fingerprints: Clean vs Lesion

Now compute the fingerprint metrics on both the original and modified k-space. The question: **does adding a lesion to the image measurably change the k-space fingerprint?**

If yes — the metrics are sensitive to local tissue changes and could serve as a detection signal.
If no — the lesion is too subtle or the metrics are too coarse to detect it at this size.

In [ ]:
profile_clean  = radial_profile(kspace_clean)
profile_lesion = radial_profile(kspace_lesion)
er_clean  = energy_ratio(kspace_clean)
er_lesion = energy_ratio(kspace_lesion)
asym_clean  = asymmetry_score(kspace_clean)
asym_lesion = asymmetry_score(kspace_lesion)

print('Fingerprint Comparison:')
print(f'  Energy ratio  — clean: {er_clean:.6f}  |  lesion: {er_lesion:.6f}  |  change: {abs(er_lesion - er_clean):.6f}')
print(f'  Asymmetry     — clean: {asym_clean:.6f}  |  lesion: {asym_lesion:.6f}  |  change: {abs(asym_lesion - asym_clean):.6f}')

min_len = min(len(profile_clean), len(profile_lesion))
profile_diff = profile_lesion[:min_len] - profile_clean[:min_len]
x = np.arange(min_len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x, profile_clean[:min_len], color='steelblue', linewidth=2, label='Clean')
axes[0].plot(x, profile_lesion[:min_len], color='tomato', linewidth=2, linestyle='--', label='With lesion')
axes[0].set_yscale('log')
axes[0].set_xlabel('Radial distance → increasing frequency')
axes[0].set_ylabel('Mean power (log scale)')
axes[0].set_title('Radial Power Profile\nClean vs Lesion')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(x, profile_diff, color='tomato', linewidth=1.5)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].fill_between(x, profile_diff, 0, where=(profile_diff > 0), alpha=0.3, color='tomato', label='Lesion adds power')
axes[1].fill_between(x, profile_diff, 0, where=(profile_diff < 0), alpha=0.3, color='steelblue', label='Lesion removes power')
axes[1].set_xlabel('Radial distance → increasing frequency')
axes[1].set_ylabel('Power difference (lesion - clean)')
axes[1].set_title('K-Space Difference\nWhich frequencies change when lesion is added?')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('K-Space Fingerprint Sensitivity to Synthetic Lesion', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../results/synthetic_fingerprint_comparison.png', dpi=150)
plt.show()

**What this demonstrates:**

The difference curve shows exactly which frequency bands change when a lesion is added to the image. Because the lesion is a smooth Gaussian blob, it contributes primarily to low-to-mid frequencies (its bulk shape) with a smaller contribution at high frequencies (its boundary). The asymmetry score changes because the lesion is placed off-center — it breaks the left-right symmetry of k-space. Both of these changes are measurable without reconstructing an image.

---

## Step 6 — Detection Threshold: How Small Can the Lesion Be?

This is the most important experiment. We sweep lesion size from large (easy to detect) to small (impossible to detect) and measure how the fingerprint change scales.

The point where the change drops to near zero is the **detection threshold** — the minimum lesion size k-space fingerprints can reliably detect. Below this threshold, the lesion is indistinguishable from noise.

In [ ]:
radii = [2, 5, 8, 12, 15, 20, 25, 30]
er_changes, asym_changes = [], []

for r in radii:
    img_modified = add_synthetic_lesion(image_clean, CENTER_Y, CENTER_X, r, INTENSITY)
    ks_modified = image_to_kspace(img_modified)
    er_changes.append(abs(energy_ratio(ks_modified) - er_clean))
    asym_changes.append(abs(asymmetry_score(ks_modified) - asym_clean))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(radii, er_changes, marker='o', color='steelblue', linewidth=2)
axes[0].set_xlabel('Lesion radius (pixels)')
axes[0].set_ylabel('Change in energy ratio')
axes[0].set_title('Energy Ratio Sensitivity vs Lesion Size\nLarger change = more detectable')
axes[0].grid(alpha=0.3)

axes[1].plot(radii, asym_changes, marker='o', color='tomato', linewidth=2)
axes[1].set_xlabel('Lesion radius (pixels)')
axes[1].set_ylabel('Change in asymmetry score')
axes[1].set_title('Asymmetry Score Sensitivity vs Lesion Size\nOff-center lesion breaks k-space symmetry')
axes[1].grid(alpha=0.3)

plt.suptitle('Detection Threshold — Minimum Lesion Size Detectable in K-Space', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../results/synthetic_detection_threshold.png', dpi=150)
plt.show()

print('Radius (px) | ΔEnergy Ratio | ΔAsymmetry')
print('-' * 45)
for r, er, asym in zip(radii, er_changes, asym_changes):
    print(f'{r:11d} | {er:.8f}  | {asym:.8f}')

**What this demonstrates:**

The curves show how fingerprint sensitivity scales with lesion size. At large radii the change is clearly detectable. As radius decreases the signal approaches zero — that inflection point is the detection threshold.

This is the ground truth validation: we know exactly what we added, we know where the threshold is, and we proved it mathematically without needing a radiologist. The next step — fastMRI+ clinical annotations — will tell us whether real pathology in this dataset falls above or below this threshold.

---

## Next Step — fastMRI+ Clinical Validation

Synthetic validation proves the method works in a controlled setting. Clinical validation proves it works on real pathology.

**fastMRI+** is an extension of the fastMRI dataset that adds radiologist-annotated findings to the same scans we already have. It maps bounding boxes, pathology labels, and clinical findings to specific `.h5` files.

To access fastMRI+:
1. Go to [https://fastmri.med.nyu.edu](https://fastmri.med.nyu.edu)
2. Submit the same data access form used for the original dataset
3. In the "intended use" field mention: *research into k-space frequency-domain analysis and pre-reconstruction diagnostic signals*
4. Download the annotation JSON files — these map to the `.h5` files you already have

With fastMRI+ labels, notebook 8 will:
- Match annotations to our `.h5` files
- Extract fingerprint metrics for labeled scans
- Train a classifier on real pathology ground truth
- Test whether the fingerprint metrics from this notebook generalize to clinical findings